# T5 Flan-small Model training and finetuning tutorial Notebook

* ### **Description** -
  #### In this notebook I have explained how to trained and finetune LLM models in healthcare doamin .This Model uses huging face API for buding Healthcare chatbot in which i have used T5-Flan-small model along with heathcare database. you can replace database of your choice to train model
  ##### - T5-Flan-Small Model -  It is LLM model specifically desing for text-to-text generation.
  ##### - TPU/GPU - I have used google colab to train this model
  ##### - HuggingFace API - I have used huggingFace API for training the model also healthcare dataset is from huggingface website 

In [ ]:
#install huggingface dataset library 
pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from datasets import load_dataset
# Acquire the training data from Hugging Face
dataset = load_dataset("iecjsu/lavita-ChatDoctor-HealthCareMagic-100k",split="train")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(…)ta-ChatDoctor-HealthCareMagic-100k.jsonl:   0%|          | 0.00/23.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19999 [00:00<?, ? examples/s]

In [ ]:
#describe data 
dataset

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 19999
})

In [ ]:
# split the dataset into train and test part
split_dataset = dataset.train_test_split(test_size=0.1)  # 10% for validation
train_dataset = split_dataset['train']
validation_dataset = split_dataset['test']

In [ ]:
#check train and test data samples records 
train_dataset, validation_dataset

(Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 17999
 }),
 Dataset({
     features: ['instruction', 'input', 'output'],
     num_rows: 2000
 }))

In [ ]:
dataset['input'][1:2]

['My baby has been pooing 5-6 times a day for a week. In the last few days it has increased to 7 and they are very watery with green stringy bits in them. He does not seem unwell i.e no temperature and still eating. He now has a very bad nappy rash from the pooing ...help!']

In [ ]:
dataset['output'][1:2]

['Hi... Thank you for consulting in Chat Doctor. It seems your kid is having viral diarrhea. Once it starts it will take 5-7 days to completely get better. Unless the kids having low urine output or very dull or excessively sleepy or blood in motion or green bilious vomiting...you need not worry. There is no need to use antibiotics unless there is blood in the motion. Antibiotics might worsen if unnecessarily used causing antibiotic associated diarrhea. I suggest you use zinc supplements (Z&D Chat Doctor.']

In [ ]:
#load tokenizer for 
from transformers import AutoTokenizer
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [ ]:
# We prefix our tasks with "answer the question"
prefix = "Please answer this question: "
# Define the preprocessing function
def preprocess_function(examples):
   """Add prefix to the sentences, tokenize the text, and set the labels"""
   # The "inputs" are the tokenized answer:
   inputs = [doc for doc in examples["input"]]
   inputs = ["question: " + q for q in examples['input']]
   model_inputs = tokenizer(inputs, padding = 'max_length',max_length=512, truncation=True)

   # The "labels" are the tokenized outputs:
   targets = [doc for doc in examples["output"] if doc is not None]
   labels = tokenizer(text_target=targets,
                      padding = 'max_length',
                      max_length=512,
                      truncation=True)

   model_inputs["labels"] = labels["input_ids"]
   return model_inputs

In [ ]:
# Map the preprocessing function across our dataset
tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_validation_dataset = validation_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/17999 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
#load the pretrain model 
from transformers import AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [ ]:
# you can enable or disable gradient checkpoint 
model.gradient_checkpointing_enable()

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,  # Reduce batch size to avoid memory issues
    gradient_accumulation_steps=8,  # Accumulate gradients
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    max_grad_norm=1.0,  # Gradient clipping
    fp16=True,  # Enable mixed precision training
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,  # Use the same dataset for simplicity
)

In [ ]:
# train the model 
trainer.train()


In [ ]:
# save the model 
model.save_pretrained("./fine-tuned-flan-t5-smallmodel")
tokenizer.save_pretrained("./fine-tuned-flan-t5-smalltokenizer")

('./fine-tuned-flan-t5-smalltokenizer/tokenizer_config.json',
 './fine-tuned-flan-t5-smalltokenizer/special_tokens_map.json',
 './fine-tuned-flan-t5-smalltokenizer/spiece.model',
 './fine-tuned-flan-t5-smalltokenizer/added_tokens.json',
 './fine-tuned-flan-t5-smalltokenizer/tokenizer.json')

In [ ]:
model_name = "./fine-tuned-flan-t5-smallmodel"  # Path to your fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("./fine-tuned-flan-t5-smalltokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

### test with sample query 

In [ ]:
# test the model with sample input data 
# Prepare a new input
input_text = "what is diarrhea"

# Tokenize the input
input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# Generate the answer using the model
output_ids = model.generate(input_ids)

# Decode the generated tokens to get the predicted answer
predicted_answer = tokenizer.decode(output_ids[0], skip_special_tokens=True)

# Print the predicted answer
print(f"Predicted answer: {predicted_answer}")  # Predicted answer: yes

Predicted answer: diarrhea (disambiguation) diarrhea is a common symptom of diarrhea.


### Integrating Langchain Framework

In [1]:
#loading pre train models  from drive
from transformers import AutoTokenizer
from transformers import AutoModelForSeq2SeqLM

from google.colab import drive
drive.mount('/content/drive')
model_name = "/content/drive/My Drive/LLMModels/fine-tuned-flan-t5-smallmodel"  # Path to your fine-tuned model
tokenizer = AutoTokenizer.from_pretrained("/content/drive/My Drive/LLMModels/fine-tuned-flan-t5-smalltokenizer")
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Mounted at /content/drive


In [ ]:
!pip install langchain_huggingface
!pip install streamlit
from langchain import LLMChain, PromptTemplate
from langchain_huggingface import HuggingFacePipeline
from transformers import pipeline

# Create a Hugging Face pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Wrap the pipeline in a LangChain HuggingFacePipeline
hf_pipeline = HuggingFacePipeline(pipeline=pipe)

# Define a prompt template
prompt_template = PromptTemplate(
    input_variables=["instruction"],
    template="Instruction: {instruction}\nResponse:"
)

# Create an LLMChain
llm_chain = LLMChain(
    llm=hf_pipeline,
    prompt=prompt_template
)


In [ ]:
### Create a Chatbot Function
#Define a function to generate responses using the LangChain pipeline:

def generate_response(instruction):
    response = llm_chain.run(instruction=instruction)
    return response

In [ ]:
# Example usage
user_input = "What are maleria?"
bot_response = generate_response(user_input)
print(bot_response)

Maleria maleria is a genus of fungi in the family Lamii


In [ ]:
def chatbot():
    print("Welcome to the Medical Chatbot! Type 'exit' to end the conversation.")
    while True:
        user_input = input("You: ")
        if user_input.lower() == "exit":
            print("Chatbot: Goodbye!")
            break
        bot_response = generate_response(user_input)
        print(f"Chatbot: {bot_response}")

# Start the chatbot
chatbot()

Welcome to the Medical Chatbot! Type 'exit' to end the conversation.
You: what is maleria?
Chatbot: Maleria Maleria is a genus of fungi in the family Ceramby
You: how to treat maleria?
Chatbot: Take a small dose of acetic acid and a small dose of acetic acid
You: what is fever?
Chatbot: Fever is a common symptom of a condition that causes a person to develop 
You: how to treate fever ?
Chatbot: Take a small amount of a small amount of a small amount of a small amount
You: i have abdominal pain and i dont understand what is it can you help me doctor ?
Chatbot: I am a doctor and I am a patient of the doctor
You: what is acne and how to treat it ?
Chatbot: acne is a condition that causes a rash, which is caused by a rash
You: how to treate acne ?
Chatbot: Use a syringe to treat acne.
You: how is exit?
Chatbot: exit the room.
You: what is capital of india


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Chatbot: India
You: exit
Chatbot: Goodbye!


### Integrating Chatbot with Streamlit , Langchain framework

In [ ]:
# Create a Hugging Face pipeline
pipe = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Wrap the pipeline in a LangChain HuggingFacePipeline
hf_pipeline = HuggingFacePipeline(pipeline=pipe)

prompt_template = PromptTemplate.from_template("You are helpful assistant. Answer the follwoing query:\n\n{user_input}")

# Create an LLMChain
llm_chain = LLMChain(
    llm=hf_pipeline,
    prompt=prompt_template
)

# Streamlit app
st.title("Healthcare Chatbot")
st.write("Ask any question and get a response from the T5 model.")

# User input
user_input = st.text_input("You: ", "")

if user_input:
    # Get response from the chatbot
    response = llm_chain.invoke({"user_input": user_input})
    st.write(f"Chatbot: {response}")

st.write("Type 'exit' to end the conversation.")